# 32 · Dynamic orchestration and retry policy

ADK 2 can run Python workflow logic through Context.run_node. Use Python loops and conditionals for orchestration; retry only operations that are safe to repeat.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** Zero model calls. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
from google.adk import Workflow, Event
from google.adk.agents.context import Context
from google.adk.workflow import node, RetryConfig

def product_lookup(node_input: str):
    return get_product(node_input)

@node(rerun_on_resume=True)
async def compare_products(ctx: Context, node_input: str):
    results = []
    for product_id in node_input.split(","):
        result = await ctx.run_node(product_lookup, product_id.strip(), run_id=product_id.strip())
        results.append(result)
    available = [p["product_id"] for p in results if p.get("stock", 0) > 0]
    return Event(message="Available product IDs: " + ", ".join(available))

workflow = Workflow(name="dynamic_catalog", edges=[("START", compare_products)])
lab = await make_lab(workflow)
await ask(lab, "P101,P102,P103")
print("RetryConfig fields:", list(RetryConfig.model_fields))


The model HTTP client already has a small retry limit in course.py. Workflow retry policy and timeout settings are independent of model retries. Avoid multiplying retry layers for paid calls. The next cell attaches the installed version's default retry policy to a read-only node.


In [ ]:
safe_lookup = node(product_lookup, name="retryable_product_lookup", retry_config=RetryConfig(), timeout=5)
print(safe_lookup.name, safe_lookup.retry_config)


## Try it

Run independent product lookups with asyncio.gather and unique run IDs. Do not reuse this retry strategy for unprotected payment operations.


## Reference

[Official ADK documentation](https://adk.dev/graphs/dynamic-workflows/). Shared configuration: `config.json`. No environment activation or login cells are needed.
